# 02 — Feature Engineering

Notebook 01 gave us a raw candidate pool (`data/processed/candidates_c81_kdlh.csv`):
230 named/unnamed landmarks along the C81 → KDLH corridor, tagged with their
OSM category and their position relative to the direct route. That's still
just "things OSM knows about near the line" — this notebook turns each
candidate into a numeric feature vector describing how *good a visual
waypoint* it plausibly is, which is what Notebook 03's model will actually
learn from.

Features built here:
1. `cross_track_nm` / `along_track_nm` — carried over from Notebook 01 (route geometry)
2. `log_size` — log-scaled footprint area
3. one-hot columns for `category` (feature type)
4. `elevation_prominence_m` — local relief from USGS 3DEP (does it stick up?)
5. `name_uniqueness` — does its name collide with another candidate on the route?
6. `nn_dist_nm` — distance to the nearest other candidate ("clutter")

Output: `data/processed/features_c81_kdlh.parquet`.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd

from vfr import pipeline

pd.set_option("display.max_columns", None)

## Step 1 — Build the feature table

The code is `vfr.pipeline.engineer_features` -- the function the DAG and
`docker compose run pipeline-processing engineer-features` run -- reading
Notebook 01's candidates and writing
`data/processed/features_c81_kdlh.parquet`, the table model-service serves
and every trainer reads. It used to be written here too, by a copy that
still fed the model three route-position columns production had dropped
as leakage. The steps below read that table back, one feature at a time.

In [2]:
out_path = pipeline.engineer_features()
df = pd.read_parquet(out_path)
print(out_path, df.shape)
df.head()

/workspace/data/processed/features_c81_kdlh.parquet (205, 21)


,osm_id,osm_type,category,name,lat,lon,cross_track_nm,along_track_nm,within_preferred_corridor,log_size,elevation_prominence_m,name_uniqueness,nn_dist_nm,category_airport,category_intersection,category_lake_or_pond,category_railroad,category_river,category_stadium,category_town,category_wind_farm
0,151331909,node,town,Superior,46.720774,-92.104080,-0.906271,315.222733,False,0.0,2.880959,1.0,0.008846,False,False,False,False,False,False,True,False
1,153420031,node,town,Montello,43.791861,-89.328819,0.062879,103.874318,True,0.0,-15.898533,1.0,0.062157,False,False,False,False,False,False,True,False
2,153546173,node,town,Round Lake,42.353355,-88.093414,0.174845,1.919655,True,0.0,6.072186,1.0,0.035830,False,False,False,False,False,False,True,False
3,153566547,node,town,Round Lake Beach,42.371688,-88.090081,0.877854,2.779455,False,0.0,-4.861698,1.0,0.731261,False,False,False,False,False,False,True,False
4,353870611,node,lake_or_pond,Big Falls Flowage,45.555885,-90.960424,-0.602629,230.635294,False,0.0,-9.940987,1.0,0.727215,False,False,True,False,False,False,False,False


## Step 2 — Size feature

`bbox_area_m2` is heavily right-skewed (a handful of real lakes vs. hundreds
of small features), which would let a couple of huge lakes dominate a
linear model's size coefficient. `log1p` compresses that range; point
features (towers, towns — no polygon, so `bbox_area_m2 == 0`) map to 0
either way.


In [3]:
df[["category", "log_size"]].groupby("category").agg(["mean", "max"])

log_size           
                   mean        max
category                          
airport        0.000000   0.000000
intersection   0.000000   0.000000
lake_or_pond  12.114577  17.443133
railroad       0.000000   0.000000
river          0.000000   0.000000
stadium       10.127442  10.127442
town           0.000000   0.000000
wind_farm     18.029248  18.029248

## Step 3 — Feature type (one-hot)

`category` is a plain string right now (`lake_or_pond`, `tower`, `town`,
...). Most models in Notebook 03 need numeric input, so one-hot encode it —
`pandas.get_dummies` is the standard tool for this on a DataFrame.


In [4]:
category_cols = [c for c in df.columns if c.startswith("category_")]
df[category_cols].sum().sort_values(ascending=False)

category_intersection    75
category_lake_or_pond    61
category_river           36
category_railroad        22
category_airport          5
category_town             4
category_stadium          1
category_wind_farm        1
dtype: int64

## Step 4 — Elevation prominence (USGS 3DEP)

A water tower on a hilltop or a bluff over a river is easier to spot from
the air than the same feature sitting on dead-flat ground. `vfr.elevation`
samples real point elevation from USGS's public Elevation Point Query
Service — at the candidate's coordinates, and at four points 1 nm out to
the N/E/S/W — and returns `candidate_elevation - mean(ring_elevation)` as a
rough local-prominence score.

The public EPQS endpoint is slow (a fraction of a second per point, times
~1,150 points for 230 candidates), so this is parallelized across threads
and cached to `data/raw/elevation_cache.csv`. **First run takes several
minutes; re-runs are instant** since every point gets cached by
coordinate.


In [5]:
df["elevation_prominence_m"].describe()

count    205.000000
mean      -4.801027
std       11.816592
min      -70.135147
25%       -9.940987
50%       -4.435078
75%        0.733433
max       44.640545
Name: elevation_prominence_m, dtype: float64

## Step 5 — Name uniqueness

A checkpoint list is only useful if a pilot can tell which "Long Lake" it
means. `name_uniqueness` is `1 / (count of candidates sharing this name)` —
1.0 for a name that appears once, 0.5 if it appears twice, NaN if the
candidate has no name at all (handled separately, not penalized the same
way as a *colliding* name).


In [6]:
df[["name", "name_uniqueness"]][df["name"].notna()].drop_duplicates().sort_values("name_uniqueness").head(10)

,name,name_uniqueness
102,CN Superior Subdivision,0.142857
69,North Fork Popple River,0.250000
82,Montello River,0.250000
98,Bark River,0.500000
172,WI 186 & WI 186;CTH C;CTH HH,0.500000
128,WI 22 & WI 22;WI 23;CTH C & WI 23;CTH C,0.500000
176,WI 33 & WI 33;CTH H,0.500000
124,WI 186 & WI 186;CTH N,0.500000
139,CTH Z & WI 13,0.500000
187,US 8 & US 8;CTH B,0.500000


## Step 6 — Nearest-neighbor clutter distance

Two candidates 0.1 nm apart are effectively the same waypoint decision —
whichever a model prefers, having its near-duplicate sitting right next to
it doesn't add information and could bias spacing logic in Notebook 07.
`nn_dist_nm` is the great-circle distance from each candidate to its
closest neighbor in the whole candidate pool (not just same-category).


In [7]:
df["nn_dist_nm"].describe()

count    205.000000
mean       0.875420
std        1.237945
min        0.002949
25%        0.225993
50%        0.479002
75%        0.899837
max        8.215631
Name: nn_dist_nm, dtype: float64

## Step 7 — The feature table

Identifying columns (name, category, coordinates) sit alongside the
numeric features — the trainers drop them before fitting a model, but
they are what labeling and reading results back need. The route-position
columns (`cross_track_nm`, `along_track_nm`, `within_preferred_corridor`)
stay for display and ordering only: `pipeline.FEATURE_COLS_BASE` says why
they are not fed to the model.

In [8]:
feature_cols = pipeline.FEATURE_COLS_BASE + category_cols
print(f"{len(feature_cols)} model features: {feature_cols}")
out_df = df

12 model features: ['log_size', 'elevation_prominence_m', 'name_uniqueness', 'nn_dist_nm', 'category_airport', 'category_intersection', 'category_lake_or_pond', 'category_railroad', 'category_river', 'category_stadium', 'category_town', 'category_wind_farm']


## Step 8 — Sanity check

A quick look at whether the features line up with intuition: prominent
towers/quarries should skew toward higher `elevation_prominence_m` than
lakes (which sit in low ground by definition), and `log_size` should
clearly separate towns/lakes from point features.


In [9]:
out_df.groupby("category")[["log_size", "elevation_prominence_m", "nn_dist_nm"]].mean().sort_values(
    "elevation_prominence_m", ascending=False
)


,log_size,elevation_prominence_m,nn_dist_nm
category,,,
airport,0.000000,8.558228,0.740129
wind_farm,18.029248,4.716911,0.642117
stadium,10.127442,1.617378,0.523446
intersection,0.000000,-1.079801,0.614849
town,0.000000,-2.951772,0.209524
lake_or_pond,12.114577,-6.520267,0.812488
railroad,0.000000,-6.980863,1.466113
river,0.000000,-10.811906,1.272969
